In [258]:
from dino import DINO_vectors_func

from PCA_kmeans import PCA_func, Kernal_PCA_func
from anomaly_photo import anomaly_photo_func

from sklearn.ensemble import IsolationForest
import numpy as np

from sklearn.svm import OneClassSVM

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

from sklearn.cluster import DBSCAN

In [259]:
def PCA_f(vector, n=25):
    scaler = StandardScaler()
    pca = PCA(n_components=n)

    X_scaled = scaler.fit_transform(vector)
    result_vectors_pca = pca.fit_transform(X_scaled)

    return result_vectors_pca
    

In [260]:
directory_all = '/Users/user/Desktop/ВКР_ИТМО/WARP_аномалии/БУТЫЛКИ ПРОЗРАЧНЫЕ 2/Все'
directory_clear = '/Users/user/Desktop/ВКР_ИТМО/WARP_аномалии/БУТЫЛКИ ПРОЗРАЧНЫЕ 2/Чистые'

glob, result_vectors = DINO_vectors_func(directory_all)
_, result_vectors_clean = DINO_vectors_func(directory_clear)

Обработано 1/134 изображений

Обработано 134/134 изображений
Получено векторов: 134
Размерность вектора: 768
Обработано 50/50 изображений
Получено векторов: 50
Размерность вектора: 768


### Предложенный метод

In [261]:
top_farthest_points_idx = PCA_func(result_vectors, n_components=25, top_k = 1000)

anomaly_photo_func(glob, top_farthest_points_idx)

Суммарная объяснённая доля для 25 компонент: 0.623
Кластеров: 2
№_2   Аномалия_3.jpg
№_7   Аномалия_1.jpg
№_11   Аномалия_5.jpg
№_17   Аномалия_4_1.jpg
№_22   Аномалия_2.jpg


### IsolationForest

In [262]:
# def isolation_forest_outliers(result_vectors, contamination=0.05):
#     iso_forest = IsolationForest(contamination=contamination, random_state=42)
#     preds = iso_forest.fit_predict(result_vectors)  # -1 = выброс
#     outlier_indices = np.where(preds == -1)[0]

#     return outlier_indices[:10] 


def isolation_forest_outliers(result_vectors, contamination=0.05):
    iso_forest = IsolationForest(contamination=contamination, random_state=42)
    iso_forest.fit(result_vectors)
    # получаем scores: чем меньше, тем более аномально
    scores = iso_forest.score_samples(result_vectors)  # массив float
    # или scores = iso_forest.decision_function(result_vectors)  # то же самое

    # сортируем индексы по возрастанию scores (самые маленькие - самые аномальные)
    ranked_indices = np.argsort(scores)  # от минимального (аном) к максимальному (норма)
    return ranked_indices  # весь ранжированный список

In [263]:
top_farthest_points_idx_ifo = isolation_forest_outliers(result_vectors, contamination=0.06)
anomaly_photo_func(glob, top_farthest_points_idx_ifo)

№_5   Аномалия_1.jpg
№_10   Аномалия_3.jpg
№_26   Аномалия_5.jpg
№_30   Аномалия_4_1.jpg
№_37   Аномалия_2.jpg


In [264]:
# c PCA
top_farthest_points_idx_ifo_pca = isolation_forest_outliers(PCA_f(result_vectors, n=25), contamination=0.05)
anomaly_photo_func(glob, top_farthest_points_idx_ifo_pca)

№_7   Аномалия_1.jpg
№_16   Аномалия_5.jpg
№_19   Аномалия_3.jpg
№_33   Аномалия_4_1.jpg
№_47   Аномалия_2.jpg


### OneClassSVM

In [265]:
# def oneclasssvm_outliers(result_vectors, contamination=0.05):
#     ocsvm = OneClassSVM(nu=contamination, kernel='rbf')
#     preds = ocsvm.fit_predict(result_vectors)  # -1 = выброс, 1 = нормальный
#     outlier_indices = np.where(preds == -1)[0]
#     return outlier_indices[:10]



def oneclasssvm_outliers(result_vectors, contamination=0.05, top_k=10):
    ocsvm = OneClassSVM(nu=contamination, kernel='rbf')
    ocsvm.fit(result_vectors)
    scores = ocsvm.decision_function(result_vectors)  # отрицательные = аномалии
    # Чем меньше score, тем аномальнее → сортируем по возрастанию
    ranked_indices = np.argsort(scores)
    return ranked_indices #, scores[ranked_indices[:top_k]]

In [266]:
# def oneclasssvm_outliers_clear(train_vectors, full_vectors, contamination=0.05):
#     ocsvm = OneClassSVM(nu=contamination, kernel='rbf')
#     ocsvm.fit(train_vectors)  # Обучение только на чистых данных
#     preds = ocsvm.predict(full_vectors)  # -1 = выброс, 1 = нормальный

#     outlier_indices = np.where(preds == -1)[0]
#     return outlier_indices[:10]


def oneclasssvm_outliers_clear(train_vectors, full_vectors, contamination=0.05, top_k=10):
    ocsvm = OneClassSVM(nu=contamination, kernel='rbf')
    ocsvm.fit(train_vectors)
    scores = ocsvm.decision_function(full_vectors)
    ranked_indices = np.argsort(scores)
    return ranked_indices

In [267]:
top_farthest_points_idx_ocs = oneclasssvm_outliers(result_vectors, contamination=0.05)
anomaly_photo_func(glob, top_farthest_points_idx_ocs)

№_7   Аномалия_4_1.jpg
№_25   Аномалия_1.jpg
№_35   Аномалия_3.jpg
№_36   Аномалия_5.jpg
№_45   Аномалия_2.jpg


In [268]:
# c PCA
top_farthest_points_idx_ocs_pca = oneclasssvm_outliers(PCA_f(result_vectors, n=25), contamination=0.05)
anomaly_photo_func(glob, top_farthest_points_idx_ocs_pca)

№_8   Аномалия_1.jpg
№_9   Аномалия_5.jpg
№_36   Аномалия_3.jpg
№_38   Аномалия_4_1.jpg
№_59   Аномалия_2.jpg


In [269]:
top_farthest_points_idx_ocs_train = oneclasssvm_outliers_clear(result_vectors_clean, result_vectors, contamination=0.06)
anomaly_photo_func(glob, top_farthest_points_idx_ocs_train)

№_15   Аномалия_3.jpg
№_23   Аномалия_1.jpg
№_24   Аномалия_5.jpg
№_27   Аномалия_4_1.jpg
№_35   Аномалия_2.jpg


In [270]:
top_farthest_points_idx_ocs_train_pca = oneclasssvm_outliers_clear(PCA_f(result_vectors_clean, n=25), PCA_f(result_vectors, n=25), contamination=0.06)
anomaly_photo_func(glob, top_farthest_points_idx_ocs_train_pca)

№_4   Аномалия_3.jpg
№_13   Аномалия_4_1.jpg
№_14   Аномалия_1.jpg
№_15   Аномалия_5.jpg
№_66   Аномалия_2.jpg


# DBSCAN

In [271]:
clustering = DBSCAN(eps=0.5, min_samples=5).fit(result_vectors)
outlier_indices = np.where(clustering.labels_ == -1)[0]

clustering.labels_

array([-1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,
       -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,
       -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,
       -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,
       -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,
       -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,
       -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,
       -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1])

In [272]:
clustering = DBSCAN(eps=10, min_samples=4).fit(PCA_f(result_vectors, n=10))
outlier_indices = np.where(clustering.labels_ == -1)[0]

clustering.labels_

array([-1, -1, -1,  0, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,  0, -1, -1,
       -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,
       -1, -1, -1, -1, -1, -1, -1, -1, -1,  1, -1, -1, -1, -1, -1, -1, -1,
       -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,  1, -1, -1, -1, -1, -1,
       -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,  1,
       -1, -1, -1, -1, -1, -1, -1,  0, -1, -1, -1, -1, -1, -1, -1, -1, -1,
       -1,  1, -1, -1, -1, -1, -1, -1, -1, -1,  1,  1,  0, -1, -1, -1, -1,
       -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,  1, -1, -1, -1])